### Split data into training, validation, and testing
(70 - 15 -15)

In [1]:
import duckdb

try:
    con.close() 
except NameError:
    pass

con = duckdb.connect("../dataset/goodreads.db") 
con.sql("SET memory_limit='4GB'")
con.sql("SET temp_directory='duckdb_tmp'")

In [2]:
#sort by hashing (for psueod random shuffle)
con.sql("""
    CREATE OR REPLACE TABLE combined_split AS
    SELECT * EXCLUDE (rn, n),
           CASE WHEN rn * 1.0 / n <= 0.70 THEN 'train'
                WHEN rn * 1.0 / n <= 0.85 THEN 'val'
                ELSE 'test' END AS split
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY hash(user_id, book_id)) AS rn,
               COUNT(*) OVER (PARTITION BY user_id) AS n
        FROM combined
    )
""")

con.sql("""
    SELECT split,
           COUNT(*) AS n_rows,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS percentage,
           COUNT(DISTINCT user_id) AS n_users
    FROM combined_split
    GROUP BY split
    ORDER BY split
""").show()

┌─────────┬────────┬────────────┬─────────┐
│  split  │ n_rows │ percentage │ n_users │
│ varchar │ int64  │   double   │  int64  │
├─────────┼────────┼────────────┼─────────┤
│ test    │ 148614 │       15.4 │    8825 │
│ train   │ 670109 │       69.6 │    8825 │
│ val     │ 144176 │       15.0 │    8825 │
└─────────┴────────┴────────────┴─────────┘



### Calculate averages from data
AFTER SPLITTING to avoid leakage

In [ ]:
#fcheck counts
con.sql("""
    CREATE OR REPLACE TABLE user_feats AS
    SELECT user_id,
           COUNT(*) AS u_n_train,
           AVG(rating) AS u_mean_rating,
           COALESCE(STDDEV_SAMP(rating), 0) AS u_std_rating
    FROM combined_split
    WHERE split = 'train'
    GROUP BY user_id
""")

con.sql("""
    CREATE OR REPLACE TABLE book_feats AS
    SELECT book_id,
           COUNT(*) AS b_n_train,
           AVG(rating) AS b_mean_rating
    FROM combined_split
    WHERE split = 'train'
    GROUP BY book_id
""")

con.sql("""
    SELECT
        (SELECT COUNT(*) FROM user_feats) AS users_with_feats,
        (SELECT COUNT(*) FROM book_feats) AS books_with_feats,
        (SELECT COUNT(DISTINCT book_id) FROM combined_split) AS books_total
""").show()

┌──────────────────┬──────────────────┬─────────────┐
│ users_with_feats │ books_with_feats │ books_total │
│      int64       │      int64       │    int64    │
├──────────────────┼──────────────────┼─────────────┤
│             8825 │            22068 │       22068 │
└──────────────────┴──────────────────┴─────────────┘



In [4]:
#use leave one out on train rows to find means
#and find correlation of each average with the rating
con.sql("""
    CREATE OR REPLACE TABLE model_data AS
    SELECT c.*,
           u.u_n_train, u.u_std_rating, b.b_n_train,

           CASE WHEN c.split = 'train' AND u.u_n_train > 1
                THEN (u.u_mean_rating * u.u_n_train - c.rating) / (u.u_n_train - 1)
                ELSE u.u_mean_rating END AS u_mean_rating,

           CASE WHEN c.split = 'train' AND b.b_n_train > 1
                THEN (b.b_mean_rating * b.b_n_train - c.rating) / (b.b_n_train - 1)
                ELSE b.b_mean_rating END AS b_mean_rating
    FROM combined_split c
    JOIN user_feats u ON c.user_id = u.user_id
    JOIN book_feats b ON c.book_id = b.book_id
""")

con.sql("""
    SELECT split,
           COUNT(*) AS n_rows,
           ROUND(CORR(u_mean_rating, rating), 3) AS corr_user_mean,
           ROUND(CORR(b_mean_rating, rating), 3) AS corr_book_mean,
           ROUND(CORR(book_avg_rating, rating), 3) AS corr_goodreads_book_avg
    FROM model_data
    GROUP BY split
    ORDER BY split
""").show()

┌─────────┬────────┬────────────────┬────────────────┬─────────────────────────┐
│  split  │ n_rows │ corr_user_mean │ corr_book_mean │ corr_goodreads_book_avg │
│ varchar │ int64  │     double     │     double     │         double          │
├─────────┼────────┼────────────────┼────────────────┼─────────────────────────┤
│ test    │ 148614 │          0.412 │          0.246 │                   0.276 │
│ train   │ 670109 │          0.409 │          0.244 │                   0.277 │
│ val     │ 144176 │          0.411 │          0.246 │                    0.28 │
└─────────┴────────┴────────────────┴────────────────┴─────────────────────────┘

